In [1]:
import numpy as np 
import pandas as pd 
import os
import numpy as np
import pandas as pd
from pathlib import Path
import os.path
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img,img_to_array
print(tf.__version__)

2.10.0


In [2]:
model = tf.keras.models.load_model('C:/Users/Rafin/OneDrive/Desktop/NSU/NSU 10th Semester/CSE299/PROJECT/MobileNet_Test/Food_FineTuned.h5')

In [3]:
# Create a list with the filepaths for training and testing
train_dir = Path(r'C:/Users/Rafin/OneDrive/Desktop/NSU/NSU 10th Semester/CSE299/PROJECT/MobileNet_Test/Bangla_Food/train')
train_filepaths = list(train_dir.glob(r'**/*.jpg'))

In [4]:
test_dir = Path(r'C:/Users/Rafin/OneDrive/Desktop/NSU/NSU 10th Semester/CSE299/PROJECT/MobileNet_Test/Bangla_Food/test')
test_filepaths = list(test_dir.glob(r'**/*.jpg'))

val_dir = Path(r'C:/Users/Rafin/OneDrive/Desktop/NSU/NSU 10th Semester/CSE299/PROJECT/MobileNet_Test/Bangla_Food/valid')
val_filepaths = list(test_dir.glob(r'**/*.jpg'))

In [5]:
def image_processing(filepath):
    labels = [str(filepath[i]).split("\\")[-2] \
        for i in range(len(filepath))]
    filepath = pd.Series(filepath, name='Filepath').astype(str)
    labels = pd.Series(labels, name='Label')
    df = pd.concat([filepath, labels], axis=1)
    df = df.sample(frac=1).reset_index(drop = True)
    return df

In [6]:
train_df = image_processing(train_filepaths)
test_df = image_processing(test_filepaths)
val_df = image_processing(val_filepaths)

In [7]:
train_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

test_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

In [8]:
train_images = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=True,
    seed=0,
    rotation_range=30,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

Found 5415 validated image filenames belonging to 58 classes.


In [9]:
val_images = train_generator.flow_from_dataframe(
    dataframe=val_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=True,
    seed=0,
    rotation_range=30,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

Found 272 validated image filenames belonging to 58 classes.


In [10]:
test_images = test_generator.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=False
)

Found 272 validated image filenames belonging to 58 classes.


In [11]:
# Predict the label of the test_images
pred = model.predict(test_images)
pred = np.argmax(pred,axis=1)
# Map the label
labels = (train_images.class_indices)
labels = dict((v,k) for k,v in labels.items())
pred1 = [labels[k] for k in pred]
pred1

9/9 [==============================] - 7s 116ms/step


['Sondesh',
 'Shik_kabab',
 'Chicken_curry',
 'Sondesh',
 'ice_cream',
 'Chicken_Grill',
 'cheesecake',
 'Noodles',
 'Chitoi  Pitha',
 'Lal-shak-Vaji',
 'Fuchka',
 'Begun Bhorta',
 'ice_cream',
 'Misti',
 'Alu Bhorta',
 'Shemai',
 'Lentil soup_Dal',
 'Lal-shak-Vaji',
 'Rosogolla',
 'Alu Bhorta',
 'Meat Curry_Gosht Bhuna',
 'Begun Bhorta',
 'ice_cream',
 'Fuchka',
 'Meat Curry_Gosht Bhuna',
 'cheesecake',
 'Chicken_curry',
 'Shawarma',
 'Chocolate_cake',
 'pizza',
 'Shemai',
 'poached_egg',
 'Misti',
 'Jalebi',
 'Shawarma',
 'Singgara',
 'Burger',
 'Chicken_wings',
 'Jorda',
 'Vegetable fritters - Beguni',
 'Chitoi  Pitha',
 'Jalebi',
 'Fried chicken - Murg Bhaja',
 'Jorda',
 'Shak-Vaji',
 'French_fries',
 'Begun Bhaja',
 'Prawn curry - Chingri bhuna',
 'Fried_rice',
 'Fish Bhuna_Mach Bhuna',
 'cheesecake',
 'Shik_kabab',
 'cup_cakes',
 'Lal-shak-Vaji',
 'Shik_kabab',
 'Momo',
 'Chicken_wings',
 'Momo',
 'Hilsha_Fish_Curry',
 'Biriyani',
 'Korola-Vaji',
 'Misti',
 'Boiled_egg',
 'Borhan

In [12]:
def output(location):
    img=load_img(location,target_size=(224,224,3))
    img=img_to_array(img)
    img=img/255
    img=np.expand_dims(img,[0])
    answer=model.predict(img)
    y_class = answer.argmax(axis=-1)
    y = " ".join(str(x) for x in y_class)
    y = int(y)
    res = labels[y]
    return res

In [15]:
img = output(r'C:/Users/Rafin/OneDrive/Desktop/NSU/NSU 10th Semester/CSE299/PROJECT/MobileNet_Test_Images/i.jpg')
img

1/1 [==============================] - 0s 21ms/step


'Fried chicken - Murg Bhaja'